# Evaluation (precision, recall, f1-score - Micro, Macro, Weighted)

## Get entities LLM

In [1]:
#%%
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from get_entities_LLM import extract_llm_entities, entity_vocab
from tqdm import tqdm

#%%
# Load your labeled dataset
df = pd.read_csv("ARP_dataset_fixed_Sentiment.csv")
df["entities"] = df["Entities"].apply(eval)

#%%
# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)
# get sentences and true entities to list
## sentences
test_sentences = test_df["Sentence"].tolist()
## entities
true_entities_list_with_sentiment = test_df["entities"].tolist()
true_entities_list = [[ent[0] for ent in entity_group if ent[0] != ''] for entity_group in true_entities_list_with_sentiment]


### no boost

In [ ]:
#%%
# Wrap original extract_llm_entities with a progress bar (no need to modify .py)
'''
def extract_llm_entities_with_progress(sentences):
    results = []
    for sent in tqdm(sentences, desc="LLM Prediction"):
        result = extract_llm_entities([sent])[0]
        results.append(result)
    return results
'''

In [7]:
#%%
# Run prediction with progress bar
'''
pred_results = extract_llm_entities_with_progress(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]
'''

'\npred_results = extract_llm_entities_with_progress(test_sentences)\npred_entities_list = [res["entities"] for res in pred_results]\n'

### boost

In [2]:
from tqdm import tqdm

BATCH_SIZE = 200  # 可視 rate limit 調 100~500

def extract_llm_entities_in_batches(sentences, batch_size=BATCH_SIZE):
    results = []
    n = len(sentences)
    for start in tqdm(range(0, n, batch_size), desc="LLM Prediction"):
        batch = sentences[start:start+batch_size]
        # 一次丟一大批，讓 extract_llm_entities 內部開並行
        results.extend(extract_llm_entities(batch))
    return results

# 使用
pred_results = extract_llm_entities_in_batches(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

LLM Prediction: 100%|██████████| 3/3 [02:06<00:00, 42.07s/it]


### evaluate

In [3]:
#%%
# Convert to multi-hot
def entities_to_multihot(entities, vocab):
    multihot = [0] * len(vocab)
    for ent in entities:
        if ent in vocab:
            multihot[vocab.index(ent)] = 1
    return multihot

y_true = [entities_to_multihot(ents, entity_vocab) for ents in true_entities_list]
y_pred = [entities_to_multihot(ents, entity_vocab) for ents in pred_entities_list]


In [5]:
#%%
# Evaluate
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=entity_vocab,
    zero_division=0
))

Micro F1: 0.6459459459459459
Macro F1: 0.5558902233354489
Weighted F1: 0.6418147380298042

Classification Report:
                           precision    recall  f1-score   support

          Federal Reserve       0.85      0.66      0.74        87
           Interest Rates       0.74      0.56      0.64        50
                Inflation       0.98      0.88      0.93        93
               Employment       0.84      0.87      0.86        31
             Unemployment       1.00      1.00      1.00         8
                      GDP       0.55      0.52      0.53        31
                    Trade       0.43      1.00      0.60         3
                 Congress       0.75      1.00      0.86         3
          Monetary Policy       0.67      0.74      0.71        66
      Financial Stability       0.25      1.00      0.40         1
          Price Stability       0.85      0.44      0.58        25
Regulatory Implementation       0.00      0.00      0.00         1
              

## Get entities nlp

In [6]:
#%%
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from get_entities_LLM import extract_llm_entities, entity_vocab
from get_entities_nlp import extract_nlp_entities
from get_entities_ft_nlp import extract_ft_entities, ENTITY_LIST
from tqdm import tqdm

#%%
# Load your labeled dataset
df = pd.read_csv("ARP_dataset_fixed_Sentiment.csv")
df["entities"] = df["Entities"].apply(eval)

#%%
# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)
# get sentences and true entities to list
## sentences
test_sentences = test_df["Sentence"].tolist()
## entities
true_entities_list_with_sentiment = test_df["entities"].tolist()
true_entities_list = [[ent[0] for ent in entity_group if ent[0] != ''] for entity_group in true_entities_list_with_sentiment]

### nlp 

In [15]:
#%%
# Wrap original extract_nlp_entities with a progress bar (no need to modify .py)
def extract_nlp_entities_with_progress(sentences):
    results = []
    for sent in tqdm(sentences, desc="Embedding-based Prediction"):
        result = extract_nlp_entities([sent])[0]
        results.append(result)
    return results

#%%
# Run prediction with progress bar
pred_results = extract_nlp_entities_with_progress(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

Embedding-based Prediction: 100%|██████████| 401/401 [02:04<00:00,  3.22it/s]


In [16]:
#%%
# Convert to multi-hot
def entities_to_multihot(entities, vocab):
    multihot = [0] * len(vocab)
    for ent in entities:
        if ent in vocab:
            multihot[vocab.index(ent)] = 1
    return multihot

y_true = [entities_to_multihot(ents, entity_vocab) for ents in true_entities_list]
y_pred = [entities_to_multihot(ents, entity_vocab) for ents in pred_entities_list]

#%%
# Evaluate
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=entity_vocab,
    zero_division=0
))

Micro F1: 0.44890726520968693
Macro F1: 0.36109856882416197
Weighted F1: 0.4828335278892033

Classification Report:
                           precision    recall  f1-score   support

          Federal Reserve       0.75      0.55      0.64        87
           Interest Rates       0.57      0.46      0.51        50
                Inflation       0.88      0.72      0.79        93
               Employment       0.29      0.16      0.21        31
             Unemployment       0.33      0.62      0.43         8
                      GDP       0.83      0.16      0.27        31
                    Trade       0.43      1.00      0.60         3
                 Congress       0.00      0.00      0.00         3
          Monetary Policy       0.45      0.70      0.54        66
      Financial Stability       0.08      1.00      0.15         1
          Price Stability       0.26      0.88      0.40        25
Regulatory Implementation       0.00      0.00      0.00         1
            

### ft nlp

In [18]:
#%%
# Wrap original extract_ft_nlp_entities with a progress bar (no need to modify .py)
def extract_ft_nlp_entities_with_progress(sentences):
    results = []
    for sent in tqdm(sentences, desc="Finetune NLP Prediction"):
        result = extract_ft_entities([sent])[0]
        results.append(result)
    return results

#%%
# Run prediction with progress bar
pred_results = extract_ft_nlp_entities_with_progress(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

Finetune NLP Prediction: 100%|██████████| 401/401 [01:02<00:00,  6.47it/s]


In [20]:
#%%
# Convert to multi-hot
def entities_to_multihot(entities, vocab):
    multihot = [0] * len(vocab)
    for ent in entities:
        if ent in vocab:
            multihot[vocab.index(ent)] = 1
    return multihot

y_true = [entities_to_multihot(ents, entity_vocab) for ents in true_entities_list]
y_pred = [entities_to_multihot(ents, entity_vocab) for ents in pred_entities_list]

#%%
# Evaluate
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=entity_vocab,
    zero_division=0
))

Micro F1: 0.7198602213162493
Macro F1: 0.5665340623143068
Weighted F1: 0.7259108594131539

Classification Report:
                           precision    recall  f1-score   support

          Federal Reserve       0.84      0.80      0.82        87
           Interest Rates       0.79      0.84      0.82        50
                Inflation       0.77      0.88      0.82        93
               Employment       0.61      0.90      0.73        31
             Unemployment       1.00      1.00      1.00         8
                      GDP       0.61      0.87      0.72        31
                    Trade       0.75      1.00      0.86         3
                 Congress       1.00      0.33      0.50         3
          Monetary Policy       0.68      0.61      0.64        66
      Financial Stability       0.00      0.00      0.00         1
          Price Stability       0.51      0.92      0.66        25
Regulatory Implementation       0.00      0.00      0.00         1
              

# Evaluation Sentiment Analysis

In [12]:
# Evaluate sentiment scoring (NLP and LLM) on the holdout set using gold entities.
# This script loads the holdout set, parses gold entity/sentiment annotations,
# runs both prediction methods, and computes evaluation metrics.

import ast
import json
import pandas as pd
import numpy as np
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    classification_report,
)

from get_sentiment_nlp import extract_nlp_sentiment
from get_sentiment_llm import extract_llm_sentiment

HOLDOUT_CSV = "holdout_eval_set.csv"

# Parse the "Entities" cell into a list of (entity, score) tuples
def parse_entities_cell(s):
    """Parse string representation of entity list into (entity, score) tuples."""
    if pd.isna(s) or s.strip() == "":
        return []
    try:
        raw = ast.literal_eval(s.strip())
    except Exception as e:
        print(f"Warning: Could not parse entities: {s[:50]}... Error: {e}")
        return []
    entities = []
    if not isinstance(raw, list):
        return []
    for item in raw:
        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue
        entity_name, score = item[0], item[1]
        if not entity_name or entity_name.strip() == "":
            continue
        try:
            score = float(score)
            entities.append((str(entity_name).strip(), score))
        except (ValueError, TypeError):
            print(f"Warning: Invalid score for entity '{entity_name}': {score}")
            continue
    return entities

# Load and prepare holdout data, returning a DataFrame of gold annotations
def load_holdout_data(csv_path):
    print(f"Loading holdout data from {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"Raw CSV shape: {df.shape}")
    expected_cols = ['Sentence', 'Entities']
    if not all(col in df.columns for col in expected_cols):
        print(f"Warning: Expected columns {expected_cols}, got {list(df.columns)}")
        # If your CSV has different column names, adjust here:
        # df = df.rename(columns={'YourSentenceCol': 'Sentence', 'YourEntitiesCol': 'Entities'})
    df = df.dropna(subset=['Sentence']).copy()
    df['Sentence'] = df['Sentence'].str.strip()
    df = df[df['Sentence'].str.len() > 0].copy()
    print(f"After cleaning: {len(df)} sentences")
    gold_data = []
    for idx, row in df.iterrows():
        sentence = row['Sentence']
        entities = parse_entities_cell(row.get('Entities', '[]'))
        if not entities:
            continue
        for pos, (entity_name, true_score) in enumerate(entities):
            gold_data.append({
                'sentence': sentence,
                'position': pos,
                'entity': entity_name,
                'true_score': true_score,
                'source_row': idx
            })
    gold_df = pd.DataFrame(gold_data)
    print(f"Gold standard: {len(gold_df)} entity annotations across {gold_df['sentence'].nunique()} sentences")
    return gold_df

# Prepare input formats for both NLP and LLM methods
def prepare_prediction_inputs(gold_df):
    # Group by sentence to get unique sentences and their entities
    sentence_groups = gold_df.groupby('sentence').apply(
        lambda x: x.sort_values('position')[['entity', 'true_score']].values.tolist()
    ).to_dict()
    print(f"Preparing inputs for {len(sentence_groups)} unique sentences")
    # LLM input: list of dicts with sentence and entities
    llm_inputs = []
    for sentence, entity_data in sentence_groups.items():
        entities = [{"name": ent} for ent, _ in entity_data]
        llm_inputs.append({
            "sentence": sentence,
            "entities": entities
        })
    # NLP input: list of (sentence, [entity names])
    nlp_inputs = []
    for sentence, entity_data in sentence_groups.items():
        entity_names = [ent for ent, _ in entity_data]
        nlp_inputs.append((sentence, entity_names))
    return llm_inputs, nlp_inputs

# Convert prediction results to a standardized DataFrame
def predictions_to_dataframe(predictions, method_name):
    rows = []
    if method_name == "LLM":
        for item in predictions:
            sentence = item["sentence"]
            for pos, entity_data in enumerate(item["entities"]):
                rows.append({
                    'sentence': sentence,
                    'position': pos,
                    'entity': entity_data["name"],
                    'pred_score': entity_data["sentiment"]
                })
    elif method_name == "NLP":
        for item in predictions:
            sentence = item["sentence"]
            for pos, entity_data in enumerate(item["entities"]):
                rows.append({
                    'sentence': sentence,
                    'position': pos,
                    'entity': entity_data["name"],
                    'pred_score': entity_data["sentiment"]
                })
    df = pd.DataFrame(rows)
    print(f"{method_name} predictions: {len(df)} entity predictions")
    return df

# Compute and print evaluation metrics for a prediction method
def compute_metrics(method_name, gold_df, pred_df, score_to_label):
    print(f"\n{'='*50}")
    print(f"EVALUATING: {method_name}")
    print(f"{'='*50}")
    merged = gold_df.merge(
        pred_df, 
        on=['sentence', 'position', 'entity'], 
        how='left',
        suffixes=('', '_pred')
    )
    print(f"Gold annotations: {len(gold_df)}")
    print(f"Predictions: {len(pred_df)}")
    print(f"Matched: {len(merged[~merged['pred_score'].isna()])}")
    print(f"Unmatched: {len(merged[merged['pred_score'].isna()])}")
    valid = merged[
        (~merged['pred_score'].isna()) & 
        (merged['pred_score'] != "") &
        (merged['pred_score'] != "")
    ].copy()
    if len(valid) == 0:
        print("No valid predictions to evaluate!")
        return
    def score_to_class(score):
        try:
            return score_to_label.get(float(score), None)
        except:
            return None
    valid['true_label'] = valid['true_score'].apply(score_to_class)
    valid['pred_label'] = valid['pred_score'].apply(score_to_class)
    valid_labels = valid[
        (~valid['true_label'].isna()) & 
        (~valid['pred_label'].isna())
    ].copy()
    if len(valid_labels) == 0:
        print("No valid label pairs after classification mapping!")
        return
    print(f"Valid for classification: {len(valid_labels)}")
    y_true = valid_labels['true_label'].astype(int).values
    y_pred = valid_labels['pred_label'].astype(int).values
    labels = sorted(set(score_to_label.values()))
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', labels=labels, zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', labels=labels, zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=labels, zero_division=0)
    y_true_reg = valid['true_score'].astype(float).values
    y_pred_reg = valid['pred_score'].astype(float).values
    mae = mean_absolute_error(y_true_reg, y_pred_reg)
    majority_score = pd.Series(y_true_reg).mode().iloc[0]
    median_score = np.median(y_true_reg)
    mae_majority = mean_absolute_error(y_true_reg, [majority_score] * len(y_true_reg))
    mae_median = mean_absolute_error(y_true_reg, [median_score] * len(y_true_reg))
    print(f"\n--- OVERALL METRICS (n={len(valid_labels)}) ---")
    print(f"Accuracy:           {accuracy:.4f}")
    print(f"Macro F1:           {macro_f1:.4f}")
    print(f"Micro F1:           {micro_f1:.4f}")
    print(f"Weighted F1:        {weighted_f1:.4f}")
    print(f"MAE:                {mae:.4f}")
    print(f"MAE (majority={majority_score:.2f}): {mae_majority:.4f} (Δ={mae_majority-mae:+.4f})")
    print(f"MAE (median={median_score:.2f}):   {mae_median:.4f} (Δ={mae_median-mae:+.4f})")
    print(f"\n--- PER-ENTITY BREAKDOWN ---")
    entity_stats = []
    for entity in sorted(valid_labels['entity'].unique()):
        entity_data = valid_labels[valid_labels['entity'] == entity]
        if len(entity_data) < 2:
            continue
        ent_y_true = entity_data['true_label'].astype(int).values
        ent_y_pred = entity_data['pred_label'].astype(int).values
        ent_acc = accuracy_score(ent_y_true, ent_y_pred)
        ent_f1 = f1_score(ent_y_true, ent_y_pred, average='macro', labels=labels, zero_division=0)
        ent_precision = precision_score(ent_y_true, ent_y_pred, average='macro', labels=labels, zero_division=0)
        ent_recall = recall_score(ent_y_true, ent_y_pred, average='macro', labels=labels, zero_division=0)
        entity_stats.append((entity, len(entity_data), ent_acc, ent_f1, ent_precision, ent_recall))
    # Sort by count descending, then F1 descending
    entity_stats.sort(key=lambda x: (-x[1], -x[3]))
    print(f"{'Entity':<25} {'n':>3}  {'acc':>6}  {'F1':>6}  {'Prec':>6}  {'Rec':>6}")
    for entity, count, acc, f1, prec, rec in entity_stats:
        print(f"{entity:<25} {count:3d}  {acc:6.3f}  {f1:6.3f}  {prec:6.3f}  {rec:6.3f}")
    print(f"\n--- CLASSIFICATION REPORT ---")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

# Main execution logic
def main():
    print("Starting Sentiment Evaluation")
    # Load score to label mapping
    try:
        with open("score_to_label.json", "r") as f:
            score_to_label_raw = json.load(f)
        score_to_label = {float(k): int(v) for k, v in score_to_label_raw.items()}
        print(f"Loaded score-to-label mapping: {len(score_to_label)} mappings")
    except FileNotFoundError:
        print("score_to_label.json not found!")
        return
    # Load holdout data
    try:
        gold_df = load_holdout_data(HOLDOUT_CSV)
    except FileNotFoundError:
        print(f"Holdout file not found: {HOLDOUT_CSV}")
        return
    if len(gold_df) == 0:
        print("No valid gold data found!")
        return
    # Prepare inputs for prediction methods
    llm_inputs, nlp_inputs = prepare_prediction_inputs(gold_df)
    # Run predictions
    print(f"\nRunning LLM predictions...")
    try:
        llm_predictions = extract_llm_sentiment(llm_inputs)
        llm_pred_df = predictions_to_dataframe(llm_predictions, "LLM")
    except Exception as e:
        print(f"LLM prediction failed: {e}")
        llm_pred_df = pd.DataFrame()
    print(f"\nRunning NLP predictions...")
    try:
        nlp_predictions = extract_nlp_sentiment(nlp_inputs)
        nlp_pred_df = predictions_to_dataframe(nlp_predictions, "NLP")
    except Exception as e:
        print(f"NLP prediction failed: {e}")
        nlp_pred_df = pd.DataFrame()
    # Evaluate both methods
    if not llm_pred_df.empty:
        compute_metrics("LLM", gold_df, llm_pred_df, score_to_label)
    if not nlp_pred_df.empty:
        compute_metrics("NLP", gold_df, nlp_pred_df, score_to_label)
    print(f"\nEvaluation complete!")

if __name__ == "__main__":
    main()

2025-08-26 16:58:50,547 | INFO | [extract] received items=284 | with_entities=284 | skipped=0
2025-08-26 16:58:50,547 | INFO | [extract] batching | chunks=15 | chunk_size≈20


Starting Sentiment Evaluation
Loaded score-to-label mapping: 7 mappings
Loading holdout data from holdout_eval_set.csv
Raw CSV shape: (400, 3)
After cleaning: 400 sentences
Gold standard: 722 entity annotations across 284 sentences
Preparing inputs for 284 unique sentences

Running LLM predictions...


2025-08-26 16:58:51,194 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 16:58:51,225 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 16:58:51,226 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 16:58:51,252 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 16:59:06,350 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 16:59:11,371 | INFO | [extract] batch failed; falling back per-item: Extra data: line 1 column 26 (char 25)
2025-08-26 16:59:11,659 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 16:59:11,683 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2025-08-26 16:59:13,006 | INFO | HTTP Request: POST https://api.deepseek.com/chat/completions "HT

LLM predictions: 722 entity predictions

Running NLP predictions...
NLP predictions: 722 entity predictions

EVALUATING: LLM
Gold annotations: 722
Predictions: 722
Matched: 684
Unmatched: 38
Valid for classification: 684

--- OVERALL METRICS (n=684) ---
Accuracy:           0.2266
Macro F1:           0.1123
Micro F1:           0.2266
Weighted F1:        0.1452
MAE:                0.3867
MAE (majority=0.33): 0.4108 (Δ=+0.0241)
MAE (median=0.33):   0.4108 (Δ=+0.0241)

--- PER-ENTITY BREAKDOWN ---
Entity                      n     acc      F1    Prec     Rec
Economic Outlook           91   0.242   0.135   0.136   0.208
Inflation                  77   0.130   0.071   0.055   0.129
Federal Reserve            74   0.230   0.053   0.033   0.143
Monetary Policy            59   0.271   0.062   0.039   0.143
Interest Rates             43   0.256   0.105   0.079   0.156
Uncertain                  29   0.276   0.105   0.085   0.186
Employment                 27   0.296   0.146   0.146   0.199
GDP  